# 06 — Model comparison and error analysis

**Question:** which model, which label scheme, and where does it fail?

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader
from smartnet.evaluation import metrics as M
from smartnet.visualization import plots as P

df = loader.load_analysis_frame()
R = config.RESULTS_DIR

In [ ]:
res5 = pd.read_csv(R/'results_motion_5cat_all.csv')
res5[res5.strategy=='grouped_event'][
    ['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean','cv_macro_auc_mean']
].round(4)

All four real models land within ~1.5 points. **Logistic regression reaches 0.934 against the random forest's 0.947** — the ensemble buys about one point. On a project where a government team maintains the result, the simpler model is genuinely competitive.

Deep learning was rejected, not overlooked: 664 independent events and 91 minority-class examples cannot train or validate a 1D CNN or LSTM credibly.

## Accuracy versus balanced accuracy

In [ ]:
fig = P.plot_accuracy_vs_balanced(res5); plt.show()

In [ ]:
cm5 = pd.read_csv(R/'confusion_motion_5cat_grouped_event.csv', index_col=0)
fig = P.plot_confusion(cm5, 'Random forest, grouped-event CV — 5 categories'); plt.show()
pd.read_csv(R/'per_class_motion_5cat_grouped_event.csv').round(3)

## Error analysis: entering versus exiting

In [ ]:
m = cm5.to_numpy()
labels = [c.replace('pred_','') for c in cm5.columns]
i_en, i_ex = labels.index('Enter'), labels.index('Exit')
en_err, ex_err = m[i_en].sum()-m[i_en,i_en], m[i_ex].sum()-m[i_ex,i_ex]
print(f'Enter: {en_err} errors, {m[i_en,i_ex]} of them predicted as Exit ({100*m[i_en,i_ex]/en_err:.0f}%)')
print(f'Exit : {ex_err} errors, {m[i_ex,i_en]} of them predicted as Enter ({100*m[i_ex,i_en]/ex_err:.0f}%)')

**Entering and exiting are confused with each other**, not with the other behaviours.

This independently reproduces the published limitation. Koudou et al. reported entry/exit confusion accounting for 83.3% and 85.7% of errors on those classes. The same failure appears here, on different data, three years later, with a richer feature set.

That convergence suggests a physical limit rather than a dataset artefact: a single accelerometer on a net's side panel registers a similar disturbance whether a body moves in or out. **Direction is largely absent from the signal**, and no model can recover information the sensor never captured.

## Collapsing the classes

In [ ]:
res4 = pd.read_csv(R/'results_motion_4cat_all.csv')
comp = pd.DataFrame({
 '5-category': res5[(res5.strategy=='grouped_event')&(res5.model=='random_forest')].iloc[0][
     ['cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean']].values,
 '4-category': res4[(res4.strategy=='grouped_event')&(res4.model=='random_forest')].iloc[0][
     ['cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean']].values,
}, index=['accuracy','balanced_accuracy','macro_f1']).astype(float).round(4)
comp

In [ ]:
fig = P.plot_per_class(pd.read_csv(R/'per_class_motion_4cat_grouped_event.csv'),
                       'Per-class performance — 4 categories (grouped-event CV)'); plt.show()

Merging entry and exit lifts sensitivity on that behaviour from 0.34/0.61 to **0.934**, and balanced accuracy from 0.787 to 0.965.

**Recommended operating point: the four-category model.** It supports counting net crossings and detecting when a net goes up or down — enough for net-use duration and timing. It does not support directional inference and should not be used for it.